In [ ]:
pip install neo4j

In [2]:
from neo4j import GraphDatabase
uri = "bolt://localhost:7687"
username = "neo4j"
password = "password123"  
driver = GraphDatabase.driver(uri, auth=(username, password))
driver.verify_connectivity()
print("Connected to Neo4j database successfully!")

Connected to Neo4j database successfully!


In [4]:
students=[
    {"name": "John Doe", "age": 20, "major": "Computer Science"},
    {"name": "Jane Smith", "age": 22, "major": "Mathematics"},
    {"name": "Alice Johnson", "age": 21, "major": "Physics"},
]

In [5]:
with driver.session() as session:
    for student in students:
        session.run(
            "CREATE (s:Student {name: $name, age: $age, major: $major})",
            name=student["name"],
            age=student["age"],
            major=student["major"]
        )
    print("Student nodes created successfully!")

Student nodes created successfully!


In [6]:
courses=[
    {"name": "Introduction to Programming", "credits": 3}, 
    {"name": "Calculus I", "credits": 4}, 
    {"name": "Physics I", "credits": 4}
    ]
with driver.session() as session:
    for course in courses:
        session.run(
            "CREATE (c:Course {name: $name, credits: $credits})",
            name=course["name"],
            credits=course["credits"]
        )
    print("Course nodes created successfully!")

Course nodes created successfully!


In [9]:
realationships=[
    { "John Doe",  "Introduction to Programming"},
    { "Jane Smith", "Calculus I"},
    { "Alice Johnson", "Physics I"},
    { "John Doe", "Calculus I"},
    { "Alice Johnson", "Introduction to Programming"},
]
with driver.session() as session:
    for student_name, course_name in realationships:
        session.run(
            "MATCH (s:Student {name: $student_name}), (c:Course {name: $course_name}) "
            "CREATE (s)-[:ENROLLED_IN]->(c)",
            student_name=student_name,
            course_name=course_name
        )
    print("Relationships created successfully!")
    

Relationships created successfully!


In [13]:
with driver.session() as session:
    result = session.run(
        "MATCH (s:Student{name:'John Doe'})-[r:ENROLLED_IN]->(c:Course) "
        "RETURN s.name AS student_name, c.name AS course_name"  
    )
    for record in result:
        print(f"{record['student_name']} is enrolled in {record['course_name']}")

John Doe is enrolled in Calculus I
John Doe is enrolled in Introduction to Programming


In [15]:

with driver.session() as session:
    result = session.run(
        """
        MATCH (s:Student)-[r:ENROLLED_IN]->(c:Course)
        WHERE s.age < 25
        RETURN s.name AS student_name, c.name AS course_name
        """
    )

    for record in result:
        print(f"{record['student_name']} is enrolled in {record['course_name']}")


John Doe is enrolled in Introduction to Programming
Jane Smith is enrolled in Calculus I
Alice Johnson is enrolled in Physics I
John Doe is enrolled in Calculus I
Alice Johnson is enrolled in Introduction to Programming


In [17]:
with driver.session() as session:
    result = session.run(
        "MATCH (s:Student{name: $name})-[r:ENROLLED_IN]->(c:Course) "
        "DELETE r",
        name="John Doe"
    )
    print("Relationships deleted successfully!")

Relationships deleted successfully!
